# Attenuator calibration and throughput noise lab

**Questions for this bench session:** where does the calibrated range end, how stable is a fixed
attenuator setting, and which measured fluctuations could be explained by its drive voltage?

The physical model is the existing **erf shutter model**. This notebook uses one SciPy optimizer.
It explores leakage-floor removal, the Chebyshev endpoint constraint, and tail selection;
it does not compare alternate physical models or reproduce the firmware optimizer.

1. [Setup and raw calibration](#setup): archive measurements before analysis.
2. [Fit and endpoint experiments](#fits): compare constraints and retain a measured operating range.
3. [Fixed-setting acquisitions](#acquire): dark, 25+35 / 25+25 / 25+35, and equal-total redistribution.
4. [Noise and throughput](#noise): timestamps, scatter, spectra, averaging, and route/error accounting.
5. [Drive-voltage evidence](#scope): local slopes, bandwidth scenarios, and simultaneous scope traces.

**Run All is offline by default.** Use the workspace `.venv` kernel. All hardware cells are opt-in.
The laser-on approval flags must remain false until the operator approves that acquisition.
No exploratory coefficients are uploaded. The optional embedded calibration does install its own
accepted fit in RAM, even with `persist=False`; it may drive the laser up to 100%.

Raw files created during acquisition live in `tools/atten_noise_data/`. Keep these with the setup
metadata: a plot or coefficient tuple alone cannot support a later refit.

## Starting evidence

For the supplied bypass experiment, take the stable laser, both isolators, 25 dB static attenuation,
and direct FVOA2-to-detector connection as the experimental setup. The apparent ~8 mV scope variation
is near the instrument floor; it is **not** an established drive-voltage RMS or bandwidth.

The earlier investigation reported the following fixed-current intervals. These are transcribed
observations, **not recalculated from the CSVs currently on disk**:

| Programmed dB | Mean net ADC (mV) | RMS scatter (mV) | Relative RMS |
|---|---:|---:|---:|
| 25 + 35, first | 145.46 | 13.77 | 9.46% |
| 25 + 25 | 1359.05 | 98.82 | 7.27% |
| 25 + 35, repeat | 129.48 | 6.70 | 5.18% |

Drift and correlation were substantial. The last two means differ by 10.50×, versus 14.74× in RMS;
shot noise alone would predict a 3.24× RMS ratio. A constant percentage fluctuation is a useful
comparison, but does not identify the component producing it.

Three quantities remain separate throughout this notebook: **temporal scatter**, **repeatability of
an average**, and **shared calibration uncertainty**. Firmware `tp_err` is an uncertainty budget,
not a measurement of illuminated temporal scatter. Its assumed 10 mV FVOA term is a hypothesis to
test, not evidence that the drive actually has that noise.

<a id="setup"></a>
## 1. Setup

Keep the launch power fixed during each illuminated sequence. Equal-total allocations compare
different local FVOA slopes at approximately the same detector level. Repeated reference settings
reveal drift and hysteresis. Record physical changes in `SETUP_NOTES` before each acquisition.

In [ ]:
from pathlib import Path
from dataclasses import asdict, is_dataclass
from datetime import datetime, timezone
import json
import math
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import optimize, signal

TOOLS = next(p.resolve() for p in (Path.cwd(), Path.cwd() / "tools", Path.cwd() / "hispec-tib/tools")
             if (p / "hispec_fibpcb.py").is_file())
sys.path.insert(0, str(TOOLS))
import hispec_fibpcb as hspcb

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.2})
DATA_DIR = TOOLS / "atten_noise_data"
BROKER, DEVICE = "hispec.caltech.edu", "hsfib-tib"
LASER, CHANNEL, OUTPUT, FIBER = "1028y", "yj", "yj_ao", "M"
CONNECT = False
FETCH_CALIBRATION = False
RUN_AUTOCAL = False
AUTOCAL_LASER_APPROVED = False  # Includes the firmware's possible 100% laser setting.
RUN_NOISE_SEQUENCE = False
NOISE_LASER_APPROVED = False
LASER_FRACTION = None  # Set an approved fixed level, 0 < value <= 1; never inferred from old data.

CALIBRATION_FILE = None  # Path to a saved calibration .npz; or fetch retained PCB records.
NOISE_CSV = None        # Path to a saved noise CSV; acquired files are selected automatically.
HISTORICAL_CSV = TOOLS / "throughput_1028y_20260915_144520.csv"
SETUP_NOTES = "1028 nm; built-in + external isolator; 25 dB static; FVOA2 directly to PD; switches bypassed"
FLASHED_ADC_SPS = None  # Enter only from a verified flashed configuration; 20 Hz stream != ADC SPS.
EXTERNAL_LASER_POWER_MW = None  # Optional simultaneous/associated power-meter value, before static loss.
STATIC_LOSS_DB, RETURN_TX = 25.0, 1.0  # Physical bypass hypothesis; no switch losses included.
EFFECTIVE_GAIN_V_PER_A = 2.0e10  # Nominal YJ ADC-input gain, including divider; use saved/live settings.

calibration = None
cal_context = {}
fit_results = {}
last_noise_csv = None
pcb = None
print("Offline defaults: no MQTT connection or hardware commands.")

### Optional connection and retained-data retrieval

Connection does not request emission. The snapshot records firmware identity, calibration, dark,
response settings and temperature. `FLASHED_ADC_SPS=None` means unknown: the historical report's
64 SPS build and the documented 250 SPS default do not establish what is running now.

Fetching retained calibration records does not start a sweep. A new sweep is a separate action below.

In [ ]:
def json_value(value):
    """Encode measurement metadata without pickle or repr-dependent parsing."""
    if is_dataclass(value):
        return json_value(asdict(value))
    if isinstance(value, dict):
        return {str(k): json_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_value(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_value(value.tolist())
    if isinstance(value, np.generic):
        return json_value(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return str(value) if isinstance(value, Path) else value


def setup_snapshot(client):
    """Read experiment context; these queries do not request laser emission."""
    return json_value(dict(utc=datetime.now(timezone.utc).isoformat(), setup=SETUP_NOTES,
        laser=LASER, channel=CHANNEL, output=OUTPUT, fiber=FIBER, flashed_adc_sps=FLASHED_ADC_SPS,
        static_loss_db=STATIC_LOSS_DB, return_tx=RETURN_TX,
        external_laser_power_mw=EXTERNAL_LASER_POWER_MW,
        status=client.status(), coeff=client.atten_coeff(LASER), atten=client.atten(LASER),
        laser_status=client.laser(LASER), laser_settings=client.laser_settings(LASER),
        pd_settings=client.pd_settings(CHANNEL), dark=client.pd_dark(CHANNEL), temps=client.temps()))


def save_calibration(dataset, context):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    path = DATA_DIR / f"cal_{LASER}_{stamp}.npz"
    np.savez_compressed(path, records=dataset.records,
                        metadata=np.array(json.dumps(json_value(dict(meta=dataset.meta, context=context)), allow_nan=False)))
    print("Raw calibration saved:", path)
    return path


def load_calibration(path):
    with np.load(path, allow_pickle=False) as archive:
        records = archive["records"].copy()
        saved = json.loads(str(archive["metadata"]))
    for item in saved["meta"]:
        if "fits" in item:
            item["fits"] = {key: hspcb.AttenuatorFitMetrics(**value) for key, value in item["fits"].items()}
    return hspcb.AttenuatorCalibrationDataset(records, tuple(saved["meta"])), saved["context"]

In [ ]:
if CONNECT:
    pcb = hspcb.HispecFibPcb(BROKER, device=DEVICE, connect=True, auto_connect=False)
    live_context = setup_snapshot(pcb)
    display(live_context)
if CALIBRATION_FILE is not None:
    calibration, cal_context = load_calibration(CALIBRATION_FILE)
if FETCH_CALIBRATION:
    if pcb is None:
        raise RuntimeError("Connect first to fetch retained calibration records.")
    if pcb.atten_calibrate().state == "running":
        raise RuntimeError("Wait for acquisition to finish before retrieving a consistent dataset.")
    calibration = pcb.atten_calibration_data(physical="all")
    cal_context = setup_snapshot(pcb)
    CALIBRATION_FILE = save_calibration(calibration, cal_context)
if calibration is not None:
    display(calibration, calibration.bridge_table())
else:
    print("No raw calibration loaded. Fit cells will skip until a saved/fetched sweep is available.")

### Optional new embedded acquisition

The firmware owns acquisition and bridging; there is no duplicate Python state machine here.
This cell changes routes, laser output, DAC positions, and possibly the installed **RAM** coefficients.
It requests `persist=False`. Review the optical setup before approving the possible full-power sweep.
On completion or interruption, it attempts laser-off and retains whatever raw records are available.
An acquisition/fit failure stays visible rather than being converted into a successful calibration.

In [ ]:
if RUN_AUTOCAL:
    if pcb is None or not AUTOCAL_LASER_APPROVED:
        raise RuntimeError("Connect and obtain operator approval for the embedded laser sweep first.")
    cal_context = setup_snapshot(pcb)
    finished = False
    try:
        state = pcb.atten_calibrate(LASER, output=OUTPUT, fiber=FIBER, dwell_ms=550, persist=False)
        deadline = time.monotonic() + 900
        while state.state == "running":
            if time.monotonic() >= deadline:
                raise TimeoutError("Calibration exceeded 15 minutes.")
            print(f"{state.physical}: {state.complete_pct:g}%", end="\r")
            time.sleep(1)
            state = pcb.atten_calibrate()
        finished = True
        display(state)
    finally:
        try:
            if not finished:
                pcb.atten_calibrate_stop()
        finally:
            try:
                pcb.laser(LASER, value=0.0)
            finally:
                calibration = pcb.atten_calibration_data(physical="all")
                CALIBRATION_FILE = save_calibration(calibration, cal_context)
else:
    print("New embedded acquisition disabled.")

<a id="fits"></a>
## 2. Fit constraints and the usable endpoint

The last measurable point is not evidence of a leakage plateau. Compare four nested experiments
using the **same normalized raw records and erf shape**:

| Experiment | Added leakage transmission | Upper correction forced to zero | Tail |
|---|---|---|---|
| Tail-mean constraint | Mean of last three usable dB points | Yes | All |
| Remove leakage | None | Yes | All |
| Free correction endpoint | None | No | All |
| Limit calibrated range | None | No | Adaptive or fixed-three exclusion |

These are exploratory fits, not firmware-verification results. The first comparison estimates the
effect of the constraint using the one host optimizer; the installed firmware curve is overlaid
separately when its coefficients were archived.

Fit in dB with fixed weights `1 / max(db_err, 0.5 dB)`. This includes the existing bridge/reference
error estimates but does **not** assume they describe illuminated scatter or independent bridge
errors. Correlated normalization and drift remain experimental limitations. Use residuals versus
segment and repeat the sweep; parameter covariance from this weighted fit is not an accuracy claim.

The six-term correction uses `x=2t-1`, with `t` measured across the fitted base-dB interval.
The old envelope is `t(1-t)`; the free-endpoint experiment uses `t`. A small fixed penalty on high
orders discourages oscillation. A dense-grid check rejects negative or nonmonotonic corrections.
The original `erf(4)` convention stays in place; nonpositive modeled transmission within the fitted
data is a failed model evaluation, not a measured saturation floor.

In [ ]:
if calibration is not None and len(calibration.records):
    for physical in ("dac1", "dac2"):
        if np.any(calibration.records.physical == physical):
            calibration.plot_physical(physical, x_axis="sweep_mv")
            plt.show()

In [ ]:
FIT_MIN_SIGMA_DB = 0.5
CHEB_TERMS, CHEB_RIDGE = 6, 1e-6
TAIL_SIGMA_MULTIPLIER, TAIL_GOOD_RUN = 3.0, 3
MIN_FIT_POINTS, MIN_FIT_SPAN_DB = 12, 10.0
GAIN_DEFAULT = hspcb.ATTENUATOR_DEFAULT_GAIN


def base_transmission(dac_mv, params, gain, leakage_db=None):
    f50, slope_per_v = params  # Scale optimizer's second parameter to order unity.
    slope = slope_per_v / 1000.0
    tx = hspcb._atten_model_tx_from_b(slope * (gain * np.asarray(dac_mv) - f50))
    opened = float(hspcb._atten_model_tx_from_b([-slope * f50])[0])
    tx = np.clip(tx / max(opened, 1e-300), 0, 1)
    if leakage_db is not None:
        floor = 10.0 ** (-leakage_db / 10.0)
        tx = floor + (1 - floor) * tx
    return tx


def correction_basis(base_db, span_db, pin_upper):
    start = hspcb.ATTENUATOR_MODEL_CORRECTION_START_DB
    t = np.maximum((np.asarray(base_db) - start) / (span_db - start), 0)
    envelope = t * (1 - t) if pin_upper else t
    return np.polynomial.chebyshev.chebvander(2 * t - 1, CHEB_TERMS - 1) * envelope[..., None]


def predict_curve(fit, dac_mv, *, corrected=True, extrapolate=False):
    x = np.asarray(dac_mv, dtype=float)
    tx = base_transmission(x, fit["params"], fit["gain"], fit["leakage_db"])
    db = -10 * np.log10(np.maximum(tx, 1e-14))  # Only a numerical guard during optimization.
    if corrected:
        db = db + correction_basis(db, fit["span_db"], fit["pin_upper"]) @ fit["cheb"]
    return np.where((tx > 0) & ((x <= fit["end_mv"]) | extrapolate), db, np.nan)


def fit_curve(points, gain, *, count=None, leakage_db=None, pin_upper=False):
    p = points[:count] if count is not None else points
    if len(p) < MIN_FIT_POINTS or np.ptp(p.db) < MIN_FIT_SPAN_DB:
        raise ValueError("Too few points or too little attenuation span after trimming.")
    x, y = np.asarray(p.sweep_mv), np.asarray(p.db)
    sigma = np.maximum(p.db_err, FIT_MIN_SIGMA_DB)
    initial = [max(float(gain * x[np.argmin(abs(y - 3.0103))]), 1.01), 2.5]
    result = optimize.least_squares(
        lambda v: (-10 * np.log10(np.maximum(base_transmission(x, v, gain, leakage_db), 1e-14)) - y) / sigma,
        initial, bounds=([1, 0.001], [2 * gain * hspcb.ATTENUATOR_DRIVE_MAX_MV, 100]),
        x_scale="jac", max_nfev=2000)
    tx = base_transmission(x, result.x, gain, leakage_db)
    if not result.success or np.any(tx <= 0):
        raise ValueError("Base fit failed or reached the erf endpoint inside measured data.")
    base = -10 * np.log10(tx)
    span = leakage_db if leakage_db is not None else float(base[-1])
    fit = dict(params=result.x, gain=gain, leakage_db=leakage_db, pin_upper=pin_upper,
               span_db=span, end_mv=float(x[-1]), count=len(p), cheb=np.zeros(CHEB_TERMS))
    basis = correction_basis(base, span, pin_upper)
    penalty = np.diag(np.sqrt(CHEB_RIDGE) * np.arange(CHEB_TERMS)**2)
    fit["cheb"] = np.linalg.lstsq(np.vstack((basis / sigma[:, None], penalty)),
                                 np.r_[(y - base) / sigma, np.zeros(CHEB_TERMS)], rcond=None)[0]
    grid = np.linspace(0, fit["end_mv"], 1000)
    trial = predict_curve(fit, grid)
    fit["correction_status"] = "accepted"
    if not np.all(np.isfinite(trial)) or np.min(trial) < -1e-5 or np.min(np.diff(trial)) < -1e-5:
        fit["cheb"][:] = 0
        fit["correction_status"] = "rejected: nonmonotonic/negative; base only"
    residual = predict_curve(fit, x) - y
    fit.update(rms_db=float(np.sqrt(np.mean(residual**2))), max_abs_db=float(max(abs(residual))),
               max_atten_db=float(predict_curve(fit, [fit["end_mv"]])[0]))
    return fit


def tail_endpoint(points, fit, multiplier=TAIL_SIGMA_MULTIPLIER, good_run=TAIL_GOOD_RUN):
    """Choose a contiguous prefix; an isolated good final residual cannot end the search."""
    residual = predict_curve(fit, points.sweep_mv) - points.db
    typical = 1.4826 * np.median(abs(residual - np.median(residual)))
    threshold = multiplier * max(float(typical), FIT_MIN_SIGMA_DB)
    good = np.isfinite(residual) & (abs(residual) <= threshold)
    for count in range(len(points), MIN_FIT_POINTS - 1, -1):
        if np.all(good[count-good_run:count]):
            return count, threshold
    raise ValueError("No acceptable tail endpoint; inspect the sweep instead of accepting a short fit.")

In [ ]:
fit_tables, endpoint_rows = [], []
if calibration is not None and len(calibration.records):
    derived = calibration.derived()
    for physical in ("dac1", "dac2"):
        points = derived[(derived.physical == physical) & derived.fit_candidate]
        points = points[np.argsort(points.sweep_mv)].view(np.recarray)
        gain = float(cal_context.get("coeff", {}).get(physical, {}).get("gain", GAIN_DEFAULT))
        if len(points) < MIN_FIT_POINTS:
            print(physical, "has too few usable points"); continue
        floor_db = float(np.mean(points.db[-3:]))
        variants = {}
        for label, floor, pinned in [("tail-mean floor", floor_db, True),
                                      ("no floor, pinned end", None, True),
                                      ("no floor, free end", None, False)]:
            try:
                variants[label] = fit_curve(points, gain, leakage_db=floor, pin_upper=pinned)
            except ValueError as exc:
                print(physical, label, exc)
        if "no floor, free end" not in variants:
            continue
        preliminary = variants["no floor, free end"]
        for label in ("drop final three", "adaptive endpoint"):
            try:
                count, threshold = ((len(points)-3, np.nan) if label == "drop final three"
                                    else tail_endpoint(points, preliminary))
                fitted = fit_curve(points, gain, count=count)
                # One refit only: freeze the preliminary threshold and explicitly validate the result.
                end_res = predict_curve(fitted, points.sweep_mv[count-TAIL_GOOD_RUN:count]) - points.db[count-TAIL_GOOD_RUN:count]
                fitted["endpoint_ok"] = bool(np.all(abs(end_res) <= threshold)) if np.isfinite(threshold) else True
                fitted["threshold_db"] = threshold
                variants[label] = fitted
            except ValueError as exc:
                print(physical, label, exc)
        fit_results[physical] = dict(points=points, variants=variants)
        for label, fit in variants.items():
            fit_tables.append(dict(physical=physical, experiment=label, points=fit["count"],
                endpoint_dac_mv=fit["end_mv"], usable_max_db=fit["max_atten_db"], rms_db=fit["rms_db"],
                worst_db=fit["max_abs_db"], correction=fit["correction_status"], endpoint_ok=fit.get("endpoint_ok", True)))
        for multiplier in (2.0, 3.0, 4.0):
            for run in (2, 3, 4):
                try:
                    count, threshold = tail_endpoint(points, preliminary, multiplier, run)
                    endpoint_rows.append(dict(physical=physical, multiplier=multiplier, good_run=run,
                        threshold_db=threshold, dropped=len(points)-count, endpoint_dac_mv=points.sweep_mv[count-1]))
                except ValueError:
                    endpoint_rows.append(dict(physical=physical, multiplier=multiplier, good_run=run, dropped=np.nan))
    display(pd.DataFrame(fit_tables), pd.DataFrame(endpoint_rows))
else:
    print("Load a raw calibration archive to compare fits and cutoff sensitivity.")

In [ ]:
for physical, result in fit_results.items():
    points, variants = result["points"], result["variants"]
    fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True, layout="constrained")
    axes[0].errorbar(points.sweep_mv, points.db, yerr=points.db_err, fmt=".", color="black", label="measured ± propagated error")
    for label, fit in variants.items():
        grid = np.linspace(0, fit["end_mv"], 600)
        line, = axes[0].plot(grid, predict_curve(fit, grid), label=label)
        residual = predict_curve(fit, points.sweep_mv, extrapolate=True) - points.db
        inside = np.arange(len(points)) < fit["count"]
        axes[1].plot(points.sweep_mv[inside], residual[inside], ".-", color=line.get_color(), label=label)
        axes[1].scatter(points.sweep_mv[~inside], residual[~inside], marker="x", color=line.get_color())
    installed = calibration._fit_coeff_for_physical(physical)
    if installed is not None:
        axes[0].plot(points.sweep_mv, hspcb._atten_db_from_coeff(installed, points.sweep_mv), "k--", label="saved firmware curve")
    full = variants["no floor, free end"]
    for corrected, label in [(False, "base only"), (True, "base + correction")]:
        axes[2].plot(points.sweep_mv, predict_curve(full, points.sweep_mv, corrected=corrected)-points.db, ".-", label=label)
    for ax in axes[1:]:
        ax.axhline(0, color="black", lw=.7)
        ax.set_ylabel("model − measured (dB)")
    for segment in np.unique(points.segment):
        x0 = points.sweep_mv[points.segment == segment][0]
        axes[2].axvline(x0, color="gray", alpha=.3)
        axes[2].text(x0, .98, f"seg {segment}", transform=axes[2].get_xaxis_transform(), fontsize=8, va="top")
    axes[0].set(title=f"{physical}: same erf shape, different endpoint assumptions", ylabel="attenuation (dB)")
    axes[0].legend(fontsize=8, ncol=2)
    selected = variants.get("adaptive endpoint")
    if selected is not None:
        for ax in axes:
            ax.axvline(selected["end_mv"], color="tab:purple", ls=":", lw=1)
        for sign in (-1, 1):
            axes[1].axhline(sign*selected["threshold_db"], color="tab:purple", ls=":", alpha=.6)
        axes[1].set_title(f"Adaptive endpoint: {selected['end_mv']:.0f} DAC mV; ±{selected['threshold_db']:.2f} dB preliminary threshold")
    axes[2].legend()
    axes[2].set_xlabel("DAC output (mV), before gain")
    plt.show()

**Read the comparisons before selecting a cutoff.** Crosses are excluded measurements evaluated by
extrapolation for diagnosis; they are not supported operating points. In-range RMS after removing
points is optimistic and is not independent validation. A later repeated sweep is the check.

The proposed adaptive rule uses `3 × max(1.4826 × MAD(residual), 0.5 dB)` and walks back until three
successive points pass. It refits once and reports whether the new endpoint still passes the frozen
threshold. The sensitivity table shows whether a small threshold/run-length change moves the endpoint
substantially. A failed endpoint check must not be used for commands.

Within this notebook only, `max_atten_db` means model dB at the accepted **DAC endpoint**. It is not
uploaded to the existing firmware, where the same name still means leakage. The optional local
candidate commands below invert the monotonic curve only inside that measured DAC interval.

<a id="acquire"></a>
## 3. Acquire fixed-setting noise data

The default program is a dark trace followed by **25+35 → 25+25 → 25+35**, with one fixed laser
level across all illuminated steps. `autolevel=False` is explicit. The alternate program redistributes
attenuation at fixed totals to change each FVOA's sensitivity without deliberately changing detector
brightness. Check the actual means: calibration errors can make equal programmed totals unequal.

An optional detector-chain control is a separate run at the **same ADC mean** with the FVOAs near
open and enough additional static attenuation to compensate. Arrange that attenuation with the laser
off and record the new setup before approving another run. This removes most FVOA slope sensitivity
without changing detector brightness; it is not an automatic step in either program.

Each record keeps its acquisition timestamp, raw/net ADC, source estimates, firmware uncertainties,
and route factors. The sidecar JSON includes command pairs, confirmed DAC positions, coefficients,
dark, response settings, temperatures, firmware ID, setup notes, and completion/error state.
Runs save after each step and again during cleanup. Laser-off is attempted on exceptions and Ctrl-C.
A kernel crash cannot run Python cleanup; a finite firmware laser auto-off is also requested.

This cell sets the selected laser to zero first and recaptures its channel's dark (RAM only).
The physical setup must be dark then; unrelated light is not controlled here. The `measure_throughput`
command still selects logical MEMS routes, which are optically bypassed in this experiment. It does
not adjust the manually set laser or DACs. Recorded route factors are audited offline below;
this notebook does not silently replace them on the PCB.

In [ ]:
PROGRAM = "repeat_levels"  # Or "redistribute".
STEP_SECONDS = 120.0
DARK_SECONDS = 60.0
SETTLE_SECONDS = 1.0         # Retain these raw samples; omit them from stationary summaries.
COMMAND_MODEL = "installed"  # Or "candidate": local voltage commands, never coefficient upload.
CANDIDATE_VARIANT = "adaptive endpoint"
PROGRAMS = {
    "repeat_levels": [("dim A", 25, 35), ("bright B", 25, 25), ("dim A repeat", 25, 35)],
    "redistribute": [("50 ref", 25, 25), ("50 left", 20, 30), ("50 ref repeat", 25, 25),
                     ("50 right", 30, 20), ("60 ref", 25, 35), ("60 balanced", 30, 30),
                     ("60 reversed", 35, 25), ("60 ref repeat", 25, 35)],
}


def command_pair(client, db1, db2):
    """Set both FVOAs explicitly; local candidate inversion is bounded by measured drive."""
    if COMMAND_MODEL == "installed":
        return client.atten(LASER, value1_db=float(db1), value2_db=float(db2))
    if COMMAND_MODEL != "candidate":
        raise ValueError("COMMAND_MODEL must be installed or candidate")
    voltages = []
    for physical, target in zip(("dac1", "dac2"), (db1, db2)):
        fit = fit_results[physical]["variants"][CANDIDATE_VARIANT]
        if not fit.get("endpoint_ok", True) or not 0 <= target <= fit["max_atten_db"]:
            raise ValueError(f"{physical}: target exceeds an accepted measured range")
        voltage = 0.0 if target == 0 else optimize.brentq(
            lambda mv: float(predict_curve(fit, [mv])[0]) - target, 0, fit["end_mv"])
        voltages.append(voltage)
    return client.atten(LASER, value1_mv=voltages[0], value2_mv=voltages[1])


def write_noise_capture(path, frames, metadata):
    """Write raw samples before downstream statistics; JSON remains ordinary inspectable metadata."""
    data = (pd.concat(frames.values(), ignore_index=True) if frames else
            pd.DataFrame(columns=[*hspcb.THROUGHPUT_DTYPE.names, "step_index", "step_label"]))
    data.to_csv(path, index=False)
    path.with_suffix(".json").write_text(json.dumps(json_value(metadata), indent=2, allow_nan=False))


def acquire_noise_sequence(client, program):
    """Own this finite bench sequence; always attempt laser-off, stream stop, and partial save."""
    if not NOISE_LASER_APPROVED or LASER_FRACTION is None or not 0 < LASER_FRACTION <= 1:
        raise RuntimeError("Approve the illuminated sequence and set its fixed LASER_FRACTION first.")
    if STEP_SECONDS < 10 or DARK_SECONDS < 10 or SETTLE_SECONDS < 0:
        raise ValueError("Use at least 10 seconds per trace and a nonnegative settling exclusion.")
    # Resolve every local inversion before any emission or DAC change.
    if COMMAND_MODEL == "candidate":
        for _, a, b in program:
            for physical, target in zip(("dac1", "dac2"), (a, b)):
                fit = fit_results[physical]["variants"][CANDIDATE_VARIANT]
                if not fit.get("endpoint_ok", True) or not 0 <= target <= fit["max_atten_db"]:
                    raise ValueError(f"{physical} cannot support {target:g} dB in the candidate range.")
    elif COMMAND_MODEL != "installed":
        raise ValueError("Unknown command model.")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    path = DATA_DIR / f"noise_{LASER}_{stamp}.csv"
    metadata = dict(context=setup_snapshot(client), program=program, command_model=COMMAND_MODEL,
        candidate_variant=CANDIDATE_VARIANT, laser_fraction=LASER_FRACTION,
        settle_seconds=SETTLE_SECONDS, steps=[], complete=False)
    if COMMAND_MODEL == "candidate":
        metadata["candidate_fits"] = {p: fit_results[p]["variants"][CANDIDATE_VARIANT] for p in ("dac1", "dac2")}
    frames, monitor, step_index = {}, None, -1
    try:
        client.stop_throughput(CHANNEL)
        client.laser(LASER, value=0.0)
        dark = client.pd_dark(CHANNEL, duration_ms=2000, persist=False)
        deadline = time.monotonic() + 15
        while dark.pending:
            if time.monotonic() > deadline:
                raise TimeoutError("Dark acquisition did not complete.")
            time.sleep(.25)
            dark = client.pd_dark(CHANNEL)
        metadata["dark_before"] = json_value(dark)
        steps = [("dark", None, None)] + list(program)
        for step_index, (label, db1, db2) in enumerate(steps):
            if db1 is not None:
                state = command_pair(client, db1, db2)
                if step_index == 1:
                    remaining_s = math.ceil(len(program) * (STEP_SECONDS + SETTLE_SECONDS + 15) + 60)
                    client.laser(LASER, value=LASER_FRACTION, autooff_s=remaining_s)
            else:
                state = client.atten(LASER)
            duration = (DARK_SECONDS if db1 is None else STEP_SECONDS) + SETTLE_SECONDS
            metadata["steps"].append(dict(index=step_index, label=label, command_db=[db1, db2],
                state=json_value(state), start_utc=datetime.now(timezone.utc).isoformat()))
            monitor = client.measure_throughput(LASER, channel=CHANNEL, output=OUTPUT, fiber=FIBER,
                autolevel=False, collect=True, off_in_s=math.ceil(duration + 15),
                max_samples=math.ceil((duration + 15) * 25))
            start, last_new, next_notice = time.monotonic(), time.monotonic(), 0.0
            previous_count = 0
            while time.monotonic() - start < duration:
                time.sleep(.25)
                rec = monitor.to_recarray()
                if len(rec) > previous_count:
                    previous_count, last_new = len(rec), time.monotonic()
                if time.monotonic() - last_new > 5:
                    raise TimeoutError("No fresh throughput records for five seconds.")
                if len(rec) and (np.any(rec.pd_mv >= hspcb.PD_ADC_USABLE_MV) or np.any(rec.autolevel)):
                    raise RuntimeError("Overrange or unexpected autolevel; raw samples retained.")
                if time.monotonic() - start >= next_notice:
                    print(f"{label}: {len(rec)} samples; {time.monotonic()-start:.0f}/{duration:.0f} s", flush=True)
                    next_notice += 15
            monitor.stop()
            frame = monitor.to_dataframe()
            frame["step_index"], frame["step_label"] = step_index, label
            frames[step_index] = frame
            monitor = None
            write_noise_capture(path, frames, metadata)
        metadata["complete"] = True
        metadata["end_context"] = setup_snapshot(client)
    except BaseException as exc:
        metadata["error"] = repr(exc)
        raise
    finally:
        metadata["laser_off_confirmed"] = False
        metadata["stream_stop_confirmed"] = False
        try:
            client.laser(LASER, value=0.0)
            metadata["laser_off_confirmed"] = True
        finally:
            try:
                if monitor is not None:
                    monitor.stop()
                else:
                    client.stop_throughput(CHANNEL)
                metadata["stream_stop_confirmed"] = True
            finally:
                if monitor is not None:
                    frame = monitor.to_dataframe()
                    frame["step_index"] = step_index
                    frame["step_label"] = metadata["steps"][-1]["label"]
                    frames[step_index] = frame
                write_noise_capture(path, frames, metadata)
                print("Saved raw/partial capture:", path)
    return path

In [ ]:
if RUN_NOISE_SEQUENCE:
    if pcb is None:
        raise RuntimeError("Connect first.")
    last_noise_csv = acquire_noise_sequence(pcb, PROGRAMS[PROGRAM])
else:
    display(pd.DataFrame(PROGRAMS[PROGRAM], columns=["step", "dac1_db", "dac2_db"]))
    print("Noise acquisition disabled. No laser or attenuator commands issued.")

### Stop selected laser / measurement

Run this cell manually if needed. It attempts the explicit laser-off even if stopping the stream fails.
It does not power down the laser bank or TECs. After an interrupted run, load the printed partial
CSV path below; it is valid evidence of the interruption, not a completed sequence.

In [ ]:
STOP_NOW = False
if STOP_NOW and pcb is not None:
    try:
        pcb.stop_throughput(CHANNEL)
    finally:
        pcb.laser(LASER, value=0.0)

<a id="noise"></a>
## 4. Load a capture and audit its assumptions

The default historical CSV is a real **autolevel** capture, not the fixed-setting data quoted above.
It is useful for route/error arithmetic and timing. Stationary-noise cells exclude autolevel records.
Select a new noise CSV to evaluate the fixed-setting experiments.

Use signed net ADC values for noise analysis. Do not fit the zero-clipped optical-power column near
darkness. Invalid and overrange samples, duplicate/reversed timestamps, gaps, channel/source changes,
DAC attenuation steps, and current steps break stationary segments; no interpolation joins them.

In [ ]:
source_csv = Path(NOISE_CSV or last_noise_csv or HISTORICAL_CSV)
noise_meta = {}
data = pd.DataFrame()
if source_csv.is_file():
    data = pd.read_csv(source_csv)
    sidecar = source_csv.with_suffix(".json")
    if sidecar.is_file():
        noise_meta = json.loads(sidecar.read_text())
    display(pd.DataFrame([dict(file=source_csv.name, records=len(data),
        columns=len(data.columns), metadata=bool(noise_meta), complete=noise_meta.get("complete", "historical/unknown"))]))
    if "pd_settings" in noise_meta.get("context", {}):
        EFFECTIVE_GAIN_V_PER_A = float(noise_meta["context"]["pd_settings"]["transimpedance_v_per_a"])
else:
    print("No capture at", source_csv, "— acquire or select a file.")

In [ ]:
def stationary_segments(frame, settle_s=SETTLE_SECONDS, minimum_s=10.0):
    """Return separate continuous fixed-setting intervals; never close gaps by dropping bad rows."""
    required = {"t_ms", "pd_net_mv", "pd_mv", "autolevel", "atten_db", "laser_current_ma", "channel", "laser"}
    if not required <= set(frame):
        raise ValueError(f"Missing current stream fields: {sorted(required-set(frame))}")
    if len(frame) < 3:
        return []
    dt = frame.t_ms.diff() / 1000.0
    nominal = float(dt[dt > 0].median())
    valid = np.isfinite(frame.pd_net_mv) & np.isfinite(frame.pd_mv) & (frame.pd_mv < hspcb.PD_ADC_USABLE_MV)
    valid &= frame.autolevel.eq(False) & np.isfinite(frame.atten_db) & np.isfinite(frame.laser_current_ma)
    if "flags" in frame:
        valid &= ~frame["flags"].astype(str).str.contains("overrange", regex=False)
    boundary = (dt <= 0) | (dt > 1.5 * nominal) | ~valid | ~valid.shift(fill_value=False)
    boundary |= frame.channel.ne(frame.channel.shift()) | frame.laser.ne(frame.laser.shift())
    boundary |= frame.atten_db.diff().abs().gt(.02) | frame.laser_current_ma.diff().abs().gt(.2)
    if "step_index" in frame:
        boundary |= frame.step_index.ne(frame.step_index.shift())
    result = []
    for _, part in frame.groupby(boundary.cumsum(), sort=False):
        if not valid.loc[part.index].all():
            continue
        elapsed = (part.t_ms - part.t_ms.iloc[0]) / 1000.0
        part = part.loc[elapsed >= settle_s].copy()
        if len(part) < 3 or (part.t_ms.iloc[-1] - part.t_ms.iloc[0]) / 1000 < minimum_s:
            continue
        result.append(part.reset_index(drop=True))
    return result


def time_statistics(part):
    y = part.pd_net_mv.to_numpy(float)
    dt = np.diff(part.t_ms.to_numpy(float)) / 1000
    rms = float(np.std(y, ddof=1))
    centered = y - y.mean()
    corr = signal.correlate(centered, centered, mode="full", method="fft")[len(y)-1:]
    corr = corr / np.arange(len(y), 0, -1)
    corr = corr / corr[0] if corr[0] > 0 else np.full_like(corr, np.nan)
    fs = 1 / np.median(dt)
    nperseg = min(1024, max(32, len(y)//4))
    f, psd = signal.welch(y, fs=fs, nperseg=nperseg, detrend="constant")
    blocks = []
    for seconds in (.05, .1, .25, .5, 1, 2, 5, 10, 20, 30):
        m = max(1, round(seconds * fs))
        n = len(y) // m
        if n < 8:
            continue
        means = y[:n*m].reshape(n, m).mean(axis=1)
        block_rms = float(np.std(means, ddof=1))
        blocks.append(dict(tau_s=m/fs, blocks=n, rms_mean_mv=block_rms,
            iid_rms_mean_mv=rms/np.sqrt(m), empirical_se_mv=block_rms/np.sqrt(n),
            adjacent_block_corr=float(np.corrcoef(means[:-1], means[1:])[0, 1]) if block_rms > 0 else np.nan,
            allan_mv=float(np.sqrt(.5*np.mean(np.diff(means)**2)))))
    total_power = np.trapezoid(psd, f)
    return dict(mean_mv=float(y.mean()), rms_mv=rms, detrended_rms_mv=float(np.std(signal.detrend(y), ddof=1)),
        relative_rms=rms/abs(y.mean()) if abs(y.mean()) > 3*rms else np.nan,
        adjacent_corr=float(corr[1]), fs_hz=fs, max_gap_ms=float(max(dt)*1000),
        timing_jitter_rms_ms=float(np.std(dt)*1000),
        sub3_fraction=float(np.trapezoid(psd[f <= 3], f[f <= 3])/total_power) if total_power > 0 else np.nan,
        acf=corr, frequencies=f, psd=psd, blocks=pd.DataFrame(blocks))


segments, statistics = [], []
if not data.empty:
    dt_ms = data.t_ms.diff()
    display(pd.DataFrame([dict(records=len(data), median_dt_ms=dt_ms[dt_ms>0].median(),
        duplicate_or_reversed=int((dt_ms<=0).sum()), gaps_over_75ms=int((dt_ms>75).sum()),
        raw_overrange=int((data.pd_mv>=hspcb.PD_ADC_USABLE_MV).sum()),
        autolevel_records=int(data.autolevel.eq(True).sum()),
        raw_code_max_error_mv=float(np.nanmax(abs(data.pd_mv-data.pd_raw*.0625))) if "pd_raw" in data else np.nan)]))
    segments = stationary_segments(data)
    statistics = [time_statistics(part) for part in segments]
    summary = pd.DataFrame([dict(interval=i, label=part.step_label.iloc[0] if "step_label" in part else "fixed interval",
        count=len(part), atten_db=part.atten_db.median(), current_ma=part.laser_current_ma.median(),
        **{k: v for k,v in stats.items() if np.isscalar(v)})
        for i,(part,stats) in enumerate(zip(segments, statistics))])
    display(summary)
    if not segments:
        print("No eligible fixed-setting intervals. Autolevel captures are not stationary noise measurements.")

In [ ]:
if not data.empty:
    t = (data.t_ms-data.t_ms.iloc[0])/1000
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(11, 8), layout="constrained")
    axes[0].plot(t, data.pd_net_mv, lw=.6)
    axes[0].set(ylabel="net ADC (mV)", title=f"{source_csv.name}: complete raw time history")
    axes[1].plot(t, data.atten_db, label="reported total attenuation")
    axes[1].set_ylabel("attenuation (dB)")
    right = axes[1].twinx()
    right.plot(t, data.laser_current_ma, color="tab:orange", alpha=.7)
    right.set_ylabel("laser current (mA)", color="tab:orange")
    axes[2].plot(t, data.pd_mv-data.pd_net_mv, label="dark subtraction")
    if "pd_net_err_mv" in data:
        axes[2].plot(t, data.pd_net_err_mv, label="reported per-reading error")
    axes[2].set(xlabel="acquisition time (s)", ylabel="mV")
    axes[2].legend()
    plt.show()

INTERVAL = 0  # Select from the summary; never concatenate separated intervals for spectra.
if statistics:
    stats, part = statistics[INTERVAL], segments[INTERVAL]
    if stats["timing_jitter_rms_ms"] > .05*1000/stats["fs_hz"]:
        print("Timing jitter exceeds 5% of cadence; FFT frequency estimates are approximate.")
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout="constrained")
    t = (part.t_ms-part.t_ms.iloc[0])/1000
    axes[0, 0].plot(t, part.pd_net_mv-stats["mean_mv"], lw=.7)
    axes[0, 0].set(xlabel="s", ylabel="deviation (mV)", title="Scatter retains drift")
    lag = np.arange(len(stats["acf"]))/stats["fs_hz"]
    axes[0, 1].plot(lag[lag<=10], stats["acf"][lag<=10])
    axes[0, 1].set(xlabel="lag (s)", ylabel="autocorrelation")
    f, psd = stats["frequencies"], stats["psd"]
    axes[1, 0].loglog(f[1:], psd[1:])
    axes[1, 0].set(xlabel="frequency (Hz)", ylabel="PSD (mV²/Hz)", title="Welch; Nyquist and aliasing limit interpretation")
    b = stats["blocks"]
    axes[1, 1].loglog(b.tau_s, b.rms_mean_mv, "o-", label="scatter of block means")
    axes[1, 1].loglog(b.tau_s, b.iid_rms_mean_mv, "--", label="independent-sample prediction")
    axes[1, 1].loglog(b.tau_s, b.allan_mv, ".-", label="nonoverlapping Allan deviation")
    axes[1, 1].set(xlabel="averaging interval (s)", ylabel="mV")
    axes[1, 1].legend(fontsize=8)
    plt.show()
    display(b)

The block table's `empirical_se_mv` divides the scatter of block means by sqrt(number of blocks).
That estimates uncertainty of the full-record mean **only if the blocks are effectively independent
and the process is sufficiently stationary**. Inspect adjacent-block correlation and stability across
block sizes; drift and too few long blocks can invalidate it. Allan deviation is another view of
averaging stability, not automatically an error bar on a mean. No calibration term is divided by √N.

The 20 Hz stream has a ~10 Hz Nyquist frequency. The nominal 20 Hz analog RC does not prevent all
aliasing. Spectra describe sampled fluctuations; they cannot recover the unsampled drive spectrum.

### Noise versus brightness and the shot-noise scale

For ADC-input effective transimpedance $G$, mean signal $V$ and equivalent noise bandwidth $B$,
the elementary photocurrent shot-noise estimate is $\sigma_V=\sqrt{2qGV B}$.
`B` is **equivalent noise bandwidth**, not necessarily an RC cutoff or the stream rate. The values
below are scenarios; filter cascades and aliasing require the real transfer functions.

Compare the repeated reference first. A variance model fitted across drifting records or different
FVOA allocations can mistake changes in the experiment for photon statistics. The reference lines
are proportionality comparisons, not a claim to have separated noise sources.

In [ ]:
ENBW_SCENARIOS_HZ = np.array([5.0, 10.0, 20.0, 40.0])
q_e = 1.602176634e-19
shot_rows = []
if statistics:
    for i, stats in enumerate(statistics):
        if stats["mean_mv"] <= 0:
            continue
        for bandwidth in ENBW_SCENARIOS_HZ:
            shot_mv = np.sqrt(2*q_e*EFFECTIVE_GAIN_V_PER_A*(stats["mean_mv"]/1000)*bandwidth)*1000
            shot_rows.append(dict(interval=i, enbw_hz=bandwidth, shot_rms_mv=shot_mv,
                                  observed_rms_mv=stats["rms_mv"], ratio=stats["rms_mv"]/shot_mv))
    display(pd.DataFrame(shot_rows))
    bright = summary.loc[summary.mean_mv > 3*summary.rms_mv]
    if not bright.empty:
        fig, ax = plt.subplots(layout="constrained")
        ax.scatter(bright.mean_mv, bright.rms_mv, label="measured intervals")
        for row in bright.itertuples():
            ax.annotate(str(row.interval), (row.mean_mv, row.rms_mv))
        mu0, rms0 = bright.mean_mv.median(), bright.rms_mv.median()
        mu = np.geomspace(max(bright.mean_mv.min()*.7, .01), bright.mean_mv.max()*1.3, 200)
        for exponent, label in [(0, "constant RMS"), (.5, "sqrt(mean) scaling"), (1, "constant fractional RMS")]:
            ax.plot(mu, rms0*(mu/mu0)**exponent, "--", label=label)
        ax.set(xscale="log", yscale="log", xlabel="mean net ADC (mV)", ylabel="RMS scatter (mV)", title="Scaling comparisons; numbers identify intervals")
        ax.legend()
        plt.show()
else:
    print("Shot-noise and brightness comparisons need fixed-setting records; no historical summary substituted.")

### Throughput and uncertainty accounting

The current stream already applies its recorded return loss to `pd_power_nw` and its recorded launch
loss to `delivered_power_nw`. First check their arithmetic, then ask whether those routes describe
the physical bypass. For the selected alternative geometry,

$$T_{bypass}=T_{recorded}\,
\frac{pd\_route\_tx}{RETURN\_TX}\,
\frac{laser\_route\_tx}{10^{-STATIC\_LOSS\_DB/10}}.$$

This is a **geometry scenario** for historical data, not a retroactive measurement of its setup.
The path factors are latched when monitoring starts. Correcting their values afterward on the PCB
does not alter an already running stream or old records.

If an associated power-meter reading is supplied, a second calculation replaces the source-power
estimate. It still depends on attenuator calibration and detector response. A stable laser can have
an uncertain absolute power estimate. No route or calibration uncertainty is invented below.

In [ ]:
audit_fields = {"tp", "tp_err", "tp_pd_err", "pd_power_nw", "pd_power_err_nw",
    "delivered_power_nw", "delivered_power_err_nw", "pd_route_tx", "laser_route_tx", "atten_tx"}
if not data.empty and audit_fields <= set(data):
    if not np.isfinite(STATIC_LOSS_DB) or not 0 < RETURN_TX <= 1:
        raise ValueError("Specify finite physical static loss and 0 < RETURN_TX <= 1.")
    d = data.loc[np.isfinite(data.delivered_power_nw) & (data.delivered_power_nw > 0)].copy()
    with np.errstate(divide="ignore", invalid="ignore"):
        reconstructed_tp = d.pd_power_nw/d.delivered_power_nw
        reconstructed_pd_err = d.pd_power_err_nw/d.delivered_power_nw
        reconstructed_err = np.hypot(reconstructed_pd_err,
            reconstructed_tp*d.delivered_power_err_nw/d.delivered_power_nw)
        route_factor = d.pd_route_tx/RETURN_TX * d.laser_route_tx/10**(-STATIC_LOSS_DB/10)
        d["tp_bypass"] = d.tp*route_factor
        d["err_bypass"] = d.tp_err*route_factor
    display(pd.DataFrame([dict(
        tp_max_abs_arithmetic_difference=float((d.tp-reconstructed_tp).abs().max()),
        tp_pd_err_max_abs_difference=float((d.tp_pd_err-reconstructed_pd_err).abs().max()),
        tp_err_max_abs_difference=float((d.tp_err-reconstructed_err).abs().max()),
        median_recorded_launch_tx=d.laser_route_tx.median(), median_recorded_return_tx=d.pd_route_tx.median(),
        bypass_multiplier=route_factor.median())]))
    fig, ax = plt.subplots(layout="constrained")
    t = (d.t_ms-data.t_ms.iloc[0])/1000
    ax.plot(t, d.tp, label="recorded throughput", alpha=.65)
    ax.plot(t, d.tp_bypass, label="25 dB/direct-return scenario" if STATIC_LOSS_DB == 25 else "selected geometry scenario")
    comparison_tx = d.atten_tx.copy()
    if "candidate_fits" in noise_meta and "step_index" in d:
        candidate = noise_meta["candidate_fits"]
        for step in noise_meta["steps"]:
            state = step["state"]
            modeled_db = sum(float(predict_curve(candidate[p], [state[key]])[0])
                             for p,key in (("dac1", "v1_mv"), ("dac2", "v2_mv")))
            comparison_tx.loc[d.step_index == step["index"]] = 10**(-modeled_db/10)
        ax.plot(t, d.tp_bypass*d.atten_tx/comparison_tx, label="geometry + experimental attenuation curve")
    if EXTERNAL_LASER_POWER_MW is not None:
        if not EXTERNAL_LASER_POWER_MW > 0:
            raise ValueError("Power-meter reading must be positive.")
        meter_denominator = EXTERNAL_LASER_POWER_MW*1e6*comparison_tx*10**(-STATIC_LOSS_DB/10)
        ax.plot(t, d.pd_power_nw*d.pd_route_tx/RETURN_TX/meter_denominator, label="geometry + supplied power-meter value")
    ax.set(yscale="symlog", ylabel="throughput ratio", xlabel="s", title="Arithmetic and optical-path assumptions are separate checks")
    ax.legend()
    plt.show()
else:
    print("Current power/error fields unavailable; route/error audit skipped.")

error_rows = []
for i,(part,stats) in enumerate(zip(segments, statistics)):
    b = stats["blocks"]
    one = b.iloc[np.argmin(abs(b.tau_s-1))] if len(b) else None
    error_rows.append(dict(interval=i, illuminated_scatter_mv=stats["rms_mv"],
        reported_pd_error_mv=part.pd_net_err_mv.median() if "pd_net_err_mv" in part else np.nan,
        block_duration_s=one.tau_s if one is not None else np.nan,
        block_mean_scatter_mv=one.rms_mean_mv if one is not None else np.nan,
        source_uncertainty_fraction=(part.delivered_power_err_nw/part.delivered_power_nw.where(part.delivered_power_nw>0)).median()
            if "delivered_power_err_nw" in part else np.nan))
if error_rows:
    display(pd.DataFrame(error_rows))
    print("Source uncertainty may include the firmware's assumed electrical-noise term; it is not all shared calibration error.")

Near darkness, tiny positive zero-clipped power or throughput is not evidence of a detection.
Use the signed voltage, a nearby dark trace, and measured averaging stability. The noise summary
suppresses fractional RMS when the mean is below three times the individual-reading scatter;
that is a descriptive reporting threshold, not a calibrated significance test for a time average.

To investigate a large absolute throughput offset, check the power meter, responsivity/effective
gain, static attenuation, and couplings after correcting geometry. The new electrical-noise term
does not correct an erroneous mean denominator.

When local candidate voltage commands were used, firmware still reports attenuation and throughput
using its installed coefficients. The additional experimental-curve line explicitly replaces that
dynamic attenuation factor offline; it is a candidate estimate, not an independent power measurement.

<a id="scope"></a>
## 5. Local slopes: how much voltage noise would matter?

All scope voltages below are **post-amplifier FVOA-control mV**, not DAC mV. Let
$s_i=dA_i/dV_i$ be dB per FVOA mV and $k=\ln(10)/10$. At frequencies where the optical response follows
the drive, the predicted fractional intensity variance is

$$\sigma_{rel}^2=k^2\left[(s_1\sigma_1)^2+(s_2\sigma_2)^2+
2\rho s_1s_2\sigma_1\sigma_2\right].$$

The equal-independent-drive inversion is only an **equivalent voltage noise**: it attributes all
remaining optical scatter to the drives. It does not measure the drives or prove causality.
Stored fit residuals describe curve uncertainty and are not added to temporal RMS here.

In [ ]:
def drive_slopes(coefficients, dac_mv, candidate_fits=None):
    slopes = []
    for physical, mv in zip(("dac1", "dac2"), dac_mv):
        coeff = coefficients[physical]
        gain = float(coeff["gain"])
        x = float(mv) + np.array([-.1, .1])/gain
        if candidate_fits is not None:
            db = predict_curve(candidate_fits[physical], x)
        else:
            db = hspcb._atten_db_from_coeff(hspcb._atten_coeff_tuple(physical, coeff), x)
        slopes.append(float((db[1]-db[0])/.2))
    return np.asarray(slopes)


equivalent_rows = []
noise_context = noise_meta.get("context", {})
coefficients = noise_context.get("coeff")
candidate_fits = noise_meta.get("candidate_fits")
dark_stats = [stats for part,stats in zip(segments, statistics)
              if "step_label" in part and part.step_label.iloc[0] == "dark"]
dark_rms = dark_stats[0]["rms_mv"] if dark_stats else np.nan
for i,(part,stats) in enumerate(zip(segments, statistics)):
    if coefficients is None or "step_index" not in part or not np.isfinite(stats["relative_rms"]):
        continue
    step = next(s for s in noise_meta["steps"] if s["index"] == int(part.step_index.iloc[0]))
    if step["label"] == "dark":
        continue
    state = step["state"]
    slopes = drive_slopes(coefficients, (state["v1_mv"], state["v2_mv"]), candidate_fits)
    scale = np.log(10)/10
    excess = np.sqrt(max(stats["rms_mv"]**2-dark_rms**2, 0)) if np.isfinite(dark_rms) else stats["rms_mv"]
    equivalent = excess/abs(stats["mean_mv"])/(scale*np.linalg.norm(slopes)) if np.linalg.norm(slopes)>0 else np.nan
    equivalent_rows.append(dict(interval=i, step=step["label"], slope1_db_per_fvoa_mv=slopes[0],
        slope2_db_per_fvoa_mv=slopes[1], equivalent_equal_independent_rms_mv=equivalent,
        dark_variance_subtracted=np.isfinite(dark_rms), observed_fraction=stats["relative_rms"],
        predicted_fraction_at_10mv=scale*10*np.linalg.norm(slopes),
        installed_curve_sigma_fraction=scale*np.hypot(coefficients["dac1"]["rms_db"], coefficients["dac2"]["rms_db"])))
if equivalent_rows:
    display(pd.DataFrame(equivalent_rows))
else:
    print("Slope/noise inference needs a capture sidecar with coefficients and actual per-FVOA DAC positions.")

# Static sensitivity scenarios. None means no actual scope RMS has been supplied.
SCOPE_RMS_MV = None  # e.g. (sigma1, sigma2), measured RMS in the stated relevant bandwidth.
DRIVE_CORRELATION = 0.0
if SCOPE_RMS_MV is not None and equivalent_rows:
    if len(SCOPE_RMS_MV) != 2 or min(SCOPE_RMS_MV) < 0 or not -1 <= DRIVE_CORRELATION <= 1:
        raise ValueError("Supply two nonnegative RMS values and a correlation from -1 to 1.")
    s = equivalent_rows[0]
    a,b = np.array([s["slope1_db_per_fvoa_mv"],s["slope2_db_per_fvoa_mv"]])*SCOPE_RMS_MV
    predicted = np.log(10)/10*np.sqrt(max(a*a+b*b+2*DRIVE_CORRELATION*a*b, 0))
    print(f"Static prediction for interval {s['interval']}: {100*predicted:.2f}% RMS; bandwidth assumptions still apply.")

### Simultaneous scope trace and bandwidth scenarios

Export one synchronous trace with these **explicit columns and units**:

| Column | Units / location |
|---|---|
| `t_s` | Seconds, strictly increasing, common clock for all scope channels |
| `fvoa1_mv`, `fvoa2_mv` | Millivolts on the two FVOA drive lines after their amplifiers |
| `pd_mv` | Millivolts at the probed photodiode analog node |

Set `SCOPE_PD_TO_ADC` to convert that node to equivalent ADC-input mV (1 for a probe at the ADC input;
use the measured divider ratio for an upstream node). Choose a matching acquired interval so its
coefficients and drive positions are used. Record the probe bandwidth, coupling, acquisition mode,
and probe-floor trace. A noisy scope channel provides an upper bound, not a deconvolved drive RMS.

The calculation retains the **complex cross spectrum** between drives. The first-order actuator
time constants below are hypothetical; the quoted 5–30/50 ms response specification is not a measured
single-pole time constant. The detector/PCB pole is also a scenario. Compare these curves with the
observed PD spectrum and drive/PD coherence before considering added analog filtering.

Compare scope and ADC only over matched stable windows. This importer does not invent synchronization
between separate scope and PCB clocks. The scope trace itself is synchronous; selecting `INTERVAL`
associates its operating point, not its timestamp alignment.

In [ ]:
SCOPE_CSV = None
SCOPE_PD_TO_ADC = 1.0
SCOPE_NOTES = "Enter probe location, bandwidth, coupling, sample rate, and instrument-floor measurement"
ACTUATOR_TAU_MS = (0.0, 10.0, 30.0, 50.0)  # Hypotheses, not converted datasheet response times.
PD_POLE_HZ = 20.0  # Hypothetical combined pole for the probed PD node.
INTEGRATE_BAND_HZ = (0.1, 10.0)

if SCOPE_CSV is not None:
    if not np.isfinite(SCOPE_PD_TO_ADC) or SCOPE_PD_TO_ADC <= 0 or PD_POLE_HZ <= 0:
        raise ValueError("Specify a positive scope-to-ADC scale and PD-pole hypothesis.")
    scope = pd.read_csv(SCOPE_CSV)
    required = {"t_s", "fvoa1_mv", "fvoa2_mv", "pd_mv"}
    if not required <= set(scope) or len(scope) < 256:
        raise ValueError("Supply at least 256 synchronous scope samples with the documented columns.")
    values = scope[list(required)].to_numpy(float)
    dt = np.diff(scope.t_s.to_numpy(float))
    if not np.all(np.isfinite(values)) or np.any(dt<=0) or np.max(abs(dt-np.median(dt))) > .01*np.median(dt):
        raise ValueError("Scope FFT requires finite, uniformly sampled data; inspect gaps instead of interpolating them.")
    if coefficients is None or not segments or "step_index" not in segments[INTERVAL]:
        raise ValueError("Load a matching capture sidecar and select its operating interval first.")
    step_id = int(segments[INTERVAL].step_index.iloc[0])
    step = next(s for s in noise_meta["steps"] if s["index"] == step_id)
    scope_dac_equivalent = [scope[column].mean()/coefficients[p]["gain"]
                           for p,column in (("dac1", "fvoa1_mv"), ("dac2", "fvoa2_mv"))]
    slopes = drive_slopes(coefficients, scope_dac_equivalent, candidate_fits)
    if not np.all(np.isfinite(slopes)):
        raise ValueError("Scope drive means lie outside the selected model's calibrated range.")
    display(pd.DataFrame(dict(physical=["dac1", "dac2"], scope_mean_fvoa_mv=[scope.fvoa1_mv.mean(), scope.fvoa2_mv.mean()],
        expected_fvoa_mv=[step["state"][key]*coefficients[p]["gain"]
                          for p,key in (("dac1", "v1_mv"), ("dac2", "v2_mv"))],
        local_db_per_fvoa_mv=slopes)))
    fs = 1/np.median(dt)
    nperseg = min(4096, len(scope)//4)  # Multiple Welch averages for interpretable coherence.
    kwargs = dict(fs=fs, nperseg=nperseg, detrend="constant")
    v1, v2 = scope.fvoa1_mv.to_numpy(), scope.fvoa2_mv.to_numpy()
    pd_scope = scope.pd_mv.to_numpy()*SCOPE_PD_TO_ADC
    f, p11 = signal.welch(v1, **kwargs)
    _, p22 = signal.welch(v2, **kwargs)
    _, p12 = signal.csd(v1, v2, **kwargs)
    _, ppd = signal.welch(pd_scope, **kwargs)
    # Coherence is insensitive to the sign; the predicted linear intensity change has a minus sign.
    predicted_static = -(slopes[0]*(v1-v1.mean()) + slopes[1]*(v2-v2.mean()))
    _, coherence = signal.coherence(predicted_static, pd_scope, fs=fs, nperseg=nperseg)
    band = (f >= INTEGRATE_BAND_HZ[0]) & (f <= INTEGRATE_BAND_HZ[1])
    if band.sum() < 3 or INTEGRATE_BAND_HZ[1] > fs/2:
        raise ValueError("Scope trace cannot resolve/cover the requested integration band.")
    pd_mean = statistics[INTERVAL]["mean_mv"]  # Matched signed net ADC mean, not scope DC offset.
    if pd_mean <= 0:
        raise ValueError("Choose an illuminated interval for fractional intensity prediction.")
    voltage_psd = np.maximum(slopes[0]**2*p11 + slopes[1]**2*p22 + 2*slopes[0]*slopes[1]*p12.real, 0)
    fig, axes = plt.subplots(2, 1, sharex=True, figsize=(11, 8), layout="constrained")
    axes[0].loglog(f[1:], ppd[1:], color="black", label="measured analog PD, ADC-equivalent")
    rows = []
    for tau_ms in ACTUATOR_TAU_MS:
        h2 = 1/(1+(2*np.pi*f*tau_ms/1000)**2) / (1+(f/PD_POLE_HZ)**2)
        predicted_psd = (pd_mean*np.log(10)/10)**2*voltage_psd*h2
        axes[0].loglog(f[1:], predicted_psd[1:], label=f"drive prediction; actuator tau={tau_ms:g} ms")
        rows.append(dict(actuator_tau_ms=tau_ms, band_low_hz=INTEGRATE_BAND_HZ[0], band_high_hz=INTEGRATE_BAND_HZ[1],
            predicted_pd_rms_mv=np.sqrt(np.trapezoid(predicted_psd[band], f[band])),
            measured_analog_pd_rms_mv=np.sqrt(np.trapezoid(ppd[band], f[band]))))
    axes[0].set(ylabel="PSD (mV²/Hz)", title="Measured spectra and assumed response filters")
    axes[0].legend(fontsize=8)
    axes[1].semilogx(f[1:], coherence[1:])
    axes[1].set(xlabel="frequency (Hz)", ylabel="drive-combination / PD coherence", ylim=(0, 1.05))
    plt.show()
    display(pd.DataFrame(rows))
    print(SCOPE_NOTES)
else:
    print("Scope analysis awaits a synchronous trace; apparent scope floor is not substituted as drive noise.")

## 6. What evidence would change the design?

| Observation | Interpretation / next check |
|---|---|
| Removing the floor fixes tail residuals on repeated raw sweeps | The tail-mean floor was constraining useful data. Compare held-out sweeps before adopting a range cap. |
| Adaptive endpoint shifts greatly with threshold or repeat | The cap is not yet established; examine bridge offsets, drift, and measurement uncertainty. |
| Candidate correction bends between sampled points | Dense-grid monotonicity rejects it; inspect the residual layer before using its inverse. |
| Equal-total allocations change fractional noise with local slopes | Consistent with drive sensitivity; compare actual detector levels and simultaneous voltage spectra. |
| Drive and analog-PD spectra are coherent at the excess-noise frequencies | Quantify gain, phase and both-drive covariance; test filtering in that band. Coherence alone does not establish causality. |
| Analog PD is quiet while ADC varies | Investigate detector-output-to-ADC path, reference, pickup and conversion configuration. |
| Analog PD varies without resolved drive variation | Bound the scope floor; investigate actuator/mechanical/thermal or detector multiplicative noise. |
| Repeated reference settings drift | Separate drift from stationary noise; use longer repeats and logged temperature before fitting a brightness law. |
| Route correction fixes the mean but not scatter | Geometry explained the scale error; noise and absolute response calibration remain separate. |

For a held-out calibration sweep, archive the first and second datasets separately and evaluate the
first fitted model against the second's derived points **within the first endpoint**, without refitting.
Repeat in both directions if hysteresis is suspected. The firmware acquisition currently sweeps one
direction; this notebook does not claim that an automatic repeat tests reverse hysteresis.

Before changing an op-amp/DAC filter, establish how much **in-band** voltage variation accounts for
the optical scatter. The present evidence motivates that measurement; it does not establish a PCB respin.

### References in this workspace

- [Current calibration and acquisition](../doc/attenuator_calibration.md)
- [Photodiode sampling and uncertainty assumptions](../doc/photodiode_notes.md)
- [Installed attenuator slope/noise calculation](../doc/api/attenuator_control.md)
- [Command/API units and side effects](../doc/commands.md)
- [Analog hardware and gain locations](../doc/hardware.md)

In [ ]:
# Optional independent repeat: reuse the original selected fits, never refit on this archive here.
VALIDATION_CALIBRATION_FILE = None
if VALIDATION_CALIBRATION_FILE is not None:
    validation, validation_context = load_calibration(VALIDATION_CALIBRATION_FILE)
    records = validation.derived()
    rows = []
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
    for physical, ax in zip(("dac1", "dac2"), axes):
        result = fit_results[physical]
        fit = result["variants"][CANDIDATE_VARIANT]
        points = records[(records.physical==physical) & records.fit_candidate & (records.sweep_mv<=fit["end_mv"])]
        if not len(points):
            continue
        residual = predict_curve(fit, points.sweep_mv)-points.db
        rows.append(dict(physical=physical, points=len(points), held_out_rms_db=np.sqrt(np.mean(residual**2)),
                         held_out_max_db=np.max(abs(residual))))
        ax.scatter(points.sweep_mv, residual, c=points.segment)
        ax.axhline(0, color="black", lw=.6)
        ax.set(title=physical, xlabel="DAC mV", ylabel="held-out model − measured (dB)")
    display(pd.DataFrame(rows))
    plt.show()